In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models as sm
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

from unet import *
from utils import *

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
BATCH_SIZE = 4
EPOCHS = 50
STEPS_PER_EPOCH = 5000
LEARNING_RATE = 3e-4
MOMENTUM = 0.5

HEIGHT = 240  # 480
WIDTH = 320  # 640
CHANNELS = 3
IMAGE_SHAPE = (HEIGHT, WIDTH, CHANNELS)
NUM_CLASSES = 8

seed = 411

DEEP_SUPERVISION = False
RGB_MODE = False

# Paths
dataset_dir = 'input/'
train_path = os.path.join(dataset_dir, 'train_val')
test_path = os.path.join(dataset_dir, 'test')

for code, name, weight in zip(class_codes, class_names, class_weights):
    print(f'Class {code}: {name} [{weight = }]')

## Preprocessing

In [ ]:
# Change directory to small example dataset for local training (without GPU)
train_path = os.path.join('input', 'example')

# Load data
train_images, val_images, test_images, train_masks, val_masks, test_masks = load_dataset(
    train_path, test_path, val_split=0.2, seed=seed
)

print('Training images:', len(train_images))
print('Validation images:', len(val_images))
print('Test images:', len(test_images))

# Check if image names match with mask names
for i in range(4):
    if i < len(train_images): print(train_images[i], '|', train_masks[i])
    if i < len(val_images): print(val_images[i], '|', val_masks[i])
    if i < len(test_images): print(test_images[i], '|', test_masks[i])

In [ ]:
# Create data generator
train_generator = DataGenerator(
    train_images, train_masks, BATCH_SIZE, HEIGHT, WIDTH,
    rgb_mask=RGB_MODE, augmentation=data_augmentation,
)
val_generator = DataGenerator(
    val_images, val_masks, BATCH_SIZE, HEIGHT, WIDTH, rgb_mask=RGB_MODE,
)
test_generator = DataGenerator(
    test_images, test_masks, BATCH_SIZE, HEIGHT, WIDTH, rgb_mask=RGB_MODE,
)

print('Number of batches in training set:', len(train_generator))
print('Number of batches in validation set:', len(val_generator))
print('Number of batches in test set:', len(test_generator))

batch = train_generator[0]
images, masks = batch
print('Training image batch shape:', images.shape)
print('Training mask batch shape:', masks.shape)

# Show some iamge-mask pairs for sanity check
for i, (img, mask) in enumerate(zip(images, masks)):
    if i > 4: break
    plt.figure(figsize=(12, 12))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.subplot(1, 2, 2)
    if RGB_MODE:  # RGB
        plt.imshow(mask)
    else:
        plt.imshow(label_to_rgb(mask, class_codes))
    plt.axis('off')
    plt.show()

## Training

In [ ]:
# SM losses
sm_dice = sm.losses.DiceLoss(class_weights=class_weights)
sm_focal = sm.losses.CategoricalFocalLoss()
sm_cce = sm.losses.CategoricalCELoss(class_weights=class_weights)
sm_jac = sm.losses.JaccardLoss(class_weights=class_weights)
sm_total_loss = (1. * sm_dice) + (3. * sm_focal) + (1. * sm_cce) + (1. * sm_jac)

# SM metrics
sm_iou = sm.metrics.IOUScore(threshold=0.5)
sm_f1 = sm.metrics.FScore(threshold=0.5)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=MOMENTUM)

model = UNet(
    input_shape=IMAGE_SHAPE,
    num_classes=3 if RGB_MODE else num_classes,  # 3 or num_classes
    num_filters=32,
    dropout=0.,
    batchnorm=True,
    final_activation='sigmoid' if RGB_MODE else 'softmax',  # sigmoid or softmax
)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy' if RGB_MODE else 'categorical_crossentropy',
    metrics=['accuracy', jaccard_coef, dice_coef, sm_iou, sm_f1],
)

# tf.keras.utils.plot_model(model, show_shapes=True)
# model.summary()

In [ ]:
EPOCHS = 3
STEPS_PER_EPOCH = 2

history = model.fit(
    train_generator, validation_data=val_generator, epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH, validation_steps=len(val_generator),
    callbacks=[
        ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1),
        EarlyStopping(monitor='val_loss', patience=10, verbose=1),
        VisualizeProgress(val_generator[0], every_n_epochs=1, num_samples=5, threshold=0.5 if RGB_MODE else None),
    ],
    verbose=1,
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 6))

plt.subplot(221)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Loss')
plt.subplot(222)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()
plt.title('Accuracy')
plt.subplot(223)
plt.plot(history.history['jaccard_coef'], label='train')
plt.plot(history.history['val_jaccard_coef'], label='val')
plt.legend()
plt.title('IoU')
plt.subplot(224)
plt.plot(history.history['dice_coef'], label='train')
plt.plot(history.history['val_dice_coef'], label='val')
plt.legend()
plt.title('Dice')

plt.tight_layout()
plt.savefig("plot-1.png")
plt.show()

In [ ]:
# save model
model.save('models/basic-unet-model.h5')

## Evaluation

In [ ]:
# load model
model = tf.keras.models.load_model('models/basic-unet-model.h5', custom_objects={
    'jaccard_coef_loss': lambda y_true, y_pred: -jaccard_coef(y_true, y_pred),
    'jaccard_coef': jaccard_coef,
    'dice_coef': dice_coef,
})

In [ ]:
# load model for testing
model = tf.keras.models.load_model('models/sm-unet-model.h5', compile=False)

In [ ]:
test = test_generator[0]

predictions = model.predict(test[0])
predictions.shape

In [ ]:
y_pred = np.argmax(predictions, axis=-1)

plt.subplots(3, 4, figsize=(12, 7))
for i, pred in enumerate(y_pred[:4]):
    img = test[0][i]
    true = test[1][i]
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.subplot(3, 4, i + 5)
    if RGB_MODE:
        plt.imshow(true)
    else:
        plt.imshow(label_to_rgb(true, class_codes, one_hot=True))
    plt.axis('off')
    plt.subplot(3, 4, i + 9)
    if RGB_MODE:
        plt.imshow(pred)
    else:
        plt.imshow(label_to_rgb(pred, class_codes, one_hot=False))
    plt.axis('off')
plt.show()